# HealthSense: Data Wrangling & Quality Assurance
### Module 1: Intelligent Health Data Ingestion, Cleaning & Transformation

This notebook demonstrates data wrangling protocols applied to medical diabetes records. We resolve biological zeroes, impute missing values, handle duplicates and outliers, and evaluate data quality before and after cleaning.


In [ ]:
import sys
from pathlib import Path
sys.path.append('..')

import pandas as pd
import numpy as np

from src.data_loader import load_default_dataset, load_dirty_sample_dataset
from src.data_cleaning import compute_data_quality_report, HealthDataWrangler
from src.feature_engineering import engineer_health_features


## 1. Load Raw Dataset
We begin by loading the canonical Pima Indians Diabetes dataset and inspect its structure.


In [ ]:
df_raw = load_default_dataset()
print('Dataset Shape:', df_raw.shape)
df_raw.head()


## 2. Automated Data Quality Audit (0-100 Score)
We assess Completeness, Validity, Consistency, and Uniqueness using the HealthSense Data Quality Engine.


In [ ]:
quality_report = compute_data_quality_report(df_raw)
print('Overall Data Quality Score:', quality_report['scores']['overall'], '/ 100')
print('Component Scores:', quality_report['scores'])
print('Explicit Missing Values:', quality_report['explicit_missing'])
print('Biological Zeros in Vital Signs:', quality_report['biological_zeros'])


## 3. Detecting Biological Zeroes
In standard diabetes records, Glucose, Blood Pressure, Skin Thickness, Insulin, and BMI cannot biologically be 0 in living humans. A zero value represents missing information.


In [ ]:
for col in ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']:
    zero_count = (df_raw[col] == 0).sum()
    print(f'{col}: {zero_count} biologically impossible zero values ({zero_count/len(df_raw):.1%})')


## 4. Full Data Wrangling Pipeline (Before vs After)
We instantiate HealthDataWrangler to replace biological zeros with NaNs, impute using median, remove duplicates, cap outliers via IQR, and verify improvements.


In [ ]:
wrangler = HealthDataWrangler(
    handle_bio_zeros=True,
    imputation_strategy='median',
    remove_duplicates=True,
    outlier_strategy='cap',
    outlier_method='iqr'
)
df_clean, audit = wrangler.fit_transform(df_raw, target_column='Outcome')

print('=== BEFORE VS AFTER DATA WRANGLING AUDIT ===')
print(audit['diff_table'].to_string(index=False))


## 5. Feature Engineering
Deriving clinically validated categorical and interaction features: BMI Categories (WHO), Age Groups, Glucose Risk Strata, and Blood Pressure Classifications.


In [ ]:
df_engineered = engineer_health_features(df_clean)
df_engineered[['BMI', 'BMI_Category', 'Glucose', 'Glucose_Risk', 'Age', 'Age_Group']].head(10)


## 6. Export Processed Data
Save the sanitized data for downstream modeling and analysis.


In [ ]:
df_clean.to_csv('../data/processed/diabetes_cleaned.csv', index=False)
print('Saved sanitized dataset to data/processed/diabetes_cleaned.csv')
